# Optional: view original Omniglot handwritten images

Unlike the `omniglot_resnet18_randomized_order_s0.h5` **feature vectors**, these are actual black-and-white image pixels. The official researchers generated their features from original Omniglot images using a pretrained ResNet18.

**Important:** This notebook shows the raw torchvision Omniglot dataset. The supplied research HDF5 has **randomly reordered class IDs** and selected exemplars, so **do not infer that a raw class index corresponds to the same numbered HDF5 class**. Use this notebook for visual understanding, not class-by-class feature alignment.

One-time setup: `python -m pip install torch torchvision`. The first run downloads Omniglot from its official torchvision source; an internet connection is required.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from torchvision.datasets import Omniglot
from torchvision.transforms import ToTensor

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
storage = root / "data" / "raw_omniglot"
background = Omniglot(root=str(storage), background=True, download=True, transform=ToTensor())
evaluation = Omniglot(root=str(storage), background=False, download=True, transform=ToTensor())
print("Background samples:", len(background))
print("Evaluation samples:", len(evaluation))
print("These are images, unlike the feature vectors in the HDF5.")

## Browse twelve characters

Change `SEED`, `N` and the `DATASET` to look around. These labels are **raw torchvision class indices**, not the randomly reordered feature classes.

In [ ]:
SEED = 42
N = 12
DATASET = background  # Change to evaluation to browse the other partition
rng = np.random.default_rng(SEED)
indexes = rng.choice(len(DATASET), size=min(N,len(DATASET)), replace=False)
cols = 4
rows = int(np.ceil(len(indexes) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(10, 2.55*rows))
for ax, idx in zip(np.asarray(axes).ravel(), indexes):
    img, label = DATASET[int(idx)]
    ax.imshow(img.squeeze().numpy(), cmap="gray")
    ax.set_title(f"Raw class {label} · item {int(idx)}", fontsize=10)
    ax.axis("off")
for ax in np.asarray(axes).ravel()[len(indexes):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

## Compare drawings from one raw class

The original study's feature array retains **multiple exemplars of each class**. See how raw drawings of a single character can vary. These are not necessarily the five preprocessed exemplars stored in the published HDF5, and this view is **not** an alignment of raw-to-feature indices.

In [ ]:
RAW_CLASS_ID = 0
# Torchvision's Omniglot keeps all targets in _characters, but using its public getitem is the robust approach.
matching = []
for idx in range(len(DATASET)):
    _, label = DATASET[idx]
    if label == RAW_CLASS_ID:
        matching.append(idx)
    if len(matching) == 10:
        break
if not matching:
    raise ValueError("Raw class not found; change RAW_CLASS_ID.")
fig, axes = plt.subplots(2, 5, figsize=(10.5, 4.7))
for ax, idx in zip(axes.ravel(), matching):
    img, _ = DATASET[idx]
    ax.imshow(img.squeeze().numpy(), cmap="gray")
    ax.set_title(f"Exemplar {idx}", fontsize=9)
    ax.axis("off")
for ax in axes.ravel()[len(matching):]:
    ax.axis("off")
fig.suptitle(f"Raw Omniglot character class {RAW_CLASS_ID}", y=1.01)
fig.tight_layout()
plt.show()

## Next step

Return to `01_explore_omniglot.ipynb` to inspect **the actual published feature file**. This optional raw-image notebook provides visual context but does not establish the mapping from raw image IDs to the randomised feature rows. To recover that correspondence exactly, you would need to reproduce the upstream extraction/reordering pipeline and record its seed/index mapping.